# 4. Singular value decomposition  [4]

TOML-MIRI — Linear Algebra for Data Science, exploration exercises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

np.set_printoptions(precision=4, suppress=True)

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

def sorted_eigh(S):
    # eigen-decomposition of a symmetric matrix, eigenvalues in DEcreasing order
    lam, V = np.linalg.eigh(S)
    idx = np.argsort(lam)[::-1]
    return lam[idx], V[:, idx]

---
## 4.1 Rotate, stretch, rotate

**Task.** For a $2\times2$ matrix with entries on sliders, show the unit circle and its image in four panels: $x$, $V^\top x$, $\Sigma V^\top x$, $U\Sigma V^\top x$. Mark $v_1, v_2$ and their images $Av_i = \sigma_i u_i$.

Points are coloured by angle so you can follow them. In the last panel the dashed circles have radius $\sigma_{\max}$ and $\sigma_{\min}$.

### Prediction (write this *before* running)

- For $A = \begin{bmatrix}1&2\\0&2\end{bmatrix}$ (Example 4.3): roughly how long is the longest semi-axis of the image ellipse?
- What does the picture look like for a singular $A$? For an orthogonal $A$?

*(fill in)*

In [ ]:
phi = np.linspace(0, 2 * np.pi, 200, endpoint=False)
circle = np.vstack([np.cos(phi), np.sin(phi)])

def svd_stages(A):
    U, s, Vt = np.linalg.svd(A)
    Sig = np.diag(s)
    stages = [np.eye(2), Vt, Sig @ Vt, U @ Sig @ Vt]
    titles = ["x  (unit circle)", "Vᵀx  (rotate/reflect)", "ΣVᵀx  (stretch)", "UΣVᵀx = Ax  (rotate/reflect)"]
    lim = max(1.3, 1.15 * s[0])
    fig, axes = plt.subplots(1, 4, figsize=(19, 4.9))
    for j, (ax, M, title) in enumerate(zip(axes, stages, titles)):
        ax.scatter(*(M @ circle), c=phi, cmap="hsv", s=6)
        for i, col in enumerate(["crimson", "royalblue"]):
            w = M @ Vt[i]                                  # image of v_i at this stage
            label = [f"v{i+1}", f"e{i+1}", f"σ{i+1}e{i+1}", f"Av{i+1} = σ{i+1}u{i+1}"][j]
            ax.annotate("", xy=w, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=col, lw=2.2))
            ax.text(*(w * 1.08 + 0.05), label, color=col)
        if j == 3:
            for r in s:
                ax.add_patch(plt.Circle((0, 0), r, fill=False, ls="--", color="gray"))
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
        ax.axhline(0, c="k", lw=0.5); ax.axvline(0, c="k", lw=0.5); ax.grid(alpha=0.3)
        ax.set_title(title)
    kappa = s[0] / s[1] if s[1] > 1e-12 * s[0] else np.inf
    fig.suptitle(f"A = {np.round(A, 3).tolist()}    σ = {s.round(4)}    "
                 f"‖A‖₂ = σ_max = {s[0]:.4f}    κ(A) = σ_max/σ_min = {kappa:.4g}")
    plt.show()
    return U, s, Vt

@interact(a11=FloatSlider(min=-3, max=3, step=0.1, value=1), a12=FloatSlider(min=-3, max=3, step=0.1, value=2),
          a21=FloatSlider(min=-3, max=3, step=0.1, value=0), a22=FloatSlider(min=-3, max=3, step=0.1, value=2))
def explore_svd(a11, a12, a21, a22):
    svd_stages(np.array([[a11, a12], [a21, a22]]))

### Verify numerically with Example 4.3

In [ ]:
A = np.array([[1.0, 2.0], [0.0, 2.0]])
U, s, Vt = np.linalg.svd(A)
V = Vt.T

check("A = U Σ Vᵀ", np.allclose(A, U @ np.diag(s) @ Vt))
check("U, V orthogonal", np.allclose(U.T @ U, np.eye(2)) and np.allclose(V.T @ V, np.eye(2)))
check("A vᵢ = σᵢ uᵢ", np.allclose(A @ V, U * s))
check("σ = (2.9208, 0.6847)  (notes)", np.allclose(s, [2.9208, 0.6847], atol=1e-4))
check("‖A‖₂ = σ_max", np.isclose(np.linalg.norm(A, 2), s[0]))
check("max over the unit circle of ‖Ax‖₂ = σ_max (brute force)", np.isclose(np.linalg.norm(A @ circle, axis=0).max(), s[0], atol=1e-3))
check("κ(A) = σ_max / σ_min", np.isclose(np.linalg.cond(A), s[0] / s[1]))
check("‖A‖₁ = 4 (max column sum),  ‖A‖_∞ = 3 (max row sum)", np.isclose(np.linalg.norm(A, 1), 4) and np.isclose(np.linalg.norm(A, np.inf), 3))
check("the maximiser of ‖Ax‖₂ is ±v₁ ≈ ±(0.2577, 0.9667)  (notes, ~3 decimals)", np.allclose(np.abs(V[:, 0]), [0.2577, 0.9667], atol=2e-3))

### Explore: Example 4.3, a singular $A$, an orthogonal $A$

In [ ]:
t = np.deg2rad(30)
examples = {"Example 4.3": [[1.0, 2.0], [0.0, 2.0]],
            "singular (rank 1)": [[1.0, 2.0], [0.5, 1.0]],
            "orthogonal (rotation by 30°)": [[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]]}
for name, M in examples.items():
    print(name)
    svd_stages(np.array(M))

---
### Explain (hand-in — in your own words)

**1. Read the spectral norm and the condition number directly from the picture [4.8, 4.9]: $\|A\|_2 = \sigma_{\max}$, $\kappa(A) = \sigma_{\max}/\sigma_{\min}$.**

*Your answer.*

**2. What happens to the ellipse (and to $\kappa$) when $A$ is singular? When $A$ is orthogonal?**

*Your answer.*

---
## 4.2 SVD vs eigendecomposition

**Task.** Compute the SVD of the $3\times2$ matrix in Example 4.1, then the eigendecompositions of $A^\top A$ and $AA^\top$.

$$A^\top A = V\Sigma^\top\Sigma V^\top, \qquad AA^\top = U\Sigma\Sigma^\top U^\top, \qquad \sigma_i^2 = \lambda_i.$$

Example 4.1: $A = \begin{bmatrix}2&1\\3&0\\1&3\end{bmatrix}$.

### Prediction (write this *before* running)

- How many eigenvalues do $A^\top A$ and $AA^\top$ have? How many are non-zero?
- Will NumPy's eigenvectors equal NumPy's singular vectors exactly?

*(fill in)*

In [ ]:
A = np.array([[2.0, 1.0], [3.0, 0.0], [1.0, 3.0]])
U, s, Vt = np.linalg.svd(A)                   # full SVD: U 3×3, Vt 2×2
V = Vt.T
lamV, EV = sorted_eigh(A.T @ A)               # 2×2
lamU, EU = sorted_eigh(A @ A.T)               # 3×3

print("singular values σ   :", s)
print("σ²                  :", s**2)
print("eigenvalues of AᵀA  :", lamV)
print("eigenvalues of AAᵀ  :", lamU)
print("\nV (SVD):\n", V, "\neigenvectors of AᵀA:\n", EV)
print("\nU (SVD):\n", U, "\neigenvectors of AAᵀ:\n", EU)

### Verify numerically — vectors are compared **up to sign**

The last checks test the values printed in the notes. A ✗ there is a finding to explain, not a bug in this notebook.

In [ ]:
def same_up_to_sign(X, Y, atol=1e-8):
    signs = np.sign(np.sum(X * Y, axis=0))
    return np.allclose(X, Y * signs, atol=atol), signs

ok_V, signs_V = same_up_to_sign(V, EV)
ok_U, signs_U = same_up_to_sign(U, EU)
check("σᵢ² = eigenvalues of AᵀA", np.allclose(s**2, lamV))
check("eigenvalues of AAᵀ = σ² plus one extra 0", np.allclose(lamU, np.r_[s**2, 0], atol=1e-10))
check(f"V = eigenvectors of AᵀA up to sign (signs {signs_V})", ok_V)
check(f"U = eigenvectors of AAᵀ up to sign (signs {signs_U})", ok_U)
check("uᵢ = A vᵢ / σᵢ  for i ≤ r", np.allclose(U[:, :2], A @ V / s))
check("u₃ spans the left null space: Aᵀ u₃ = 0", np.allclose(A.T @ U[:, 2], 0))

# the values printed in the notes (full SVD)
U_notes = np.array([[-0.531, -0.113, -0.839], [-0.595, -0.654, 0.466], [-0.602, 0.748, 0.279]])
s_notes = np.array([4.169, 2.572])
Vt_notes_full = np.array([[-0.828, -0.561], [-0.561, 0.828]])
Vt_notes_econ = np.array([[-0.560, 0.828], [0.828, 0.56]])
check("σ match the notes", np.allclose(s, s_notes, atol=1e-3))
check("U matches the notes up to sign", same_up_to_sign(U, U_notes, atol=2e-3)[0])
check("notes' full SVD reconstructs A", np.allclose(U_notes[:, :2] @ np.diag(s_notes) @ Vt_notes_full, A, atol=1e-2))
check("notes' economy SVD reconstructs A", np.allclose(U_notes[:, :2] @ np.diag(s_notes) @ Vt_notes_econ, A, atol=1e-2))

### Explore: flip signs

In [ ]:
U2, V2 = U.copy(), V.copy()
U2[:, 0] *= -1; V2[:, 0] *= -1
check("flipping u₁ AND v₁ still gives A", np.allclose(U2[:, :2] @ np.diag(s) @ V2.T, A))
U3 = U.copy(); U3[:, 0] *= -1
check("flipping only u₁ does NOT give A", not np.allclose(U3[:, :2] @ np.diag(s) @ V.T, A))

---
### Explain (hand-in — in your own words)

**1. Match $V$, $U$ and $\sigma_i^2$ with the eigen-results [4.2].**

*Your answer.*

**2. Why does $AA^\top$ have an extra zero eigenvalue?**
*(Hint: what are the sizes and ranks of $A^\top A$ and $AA^\top$? Which subspace does the extra eigenvector span?)*

*Your answer.*

**3. Are the singular vectors unique (look at their signs)? The notes print two different $V^\top$ for the full and economy SVD. Are both valid?**
*(Hint: use the sign-flip cell and the reconstruction checks.)*

*Your answer.*

---
## 4.3 Low-rank approximation of an image

**Task.** Load a grayscale image as a matrix, plot its singular values on a log scale, and show the rank-$k$ approximations for $k = 1, 5, 20, 50, 100$ next to the original.

$$A_k = \sum_{i=1}^k \sigma_i u_i v_i^\top, \qquad \|A - A_k\|_2 = \sigma_{k+1}, \qquad \|A - A_k\|_F = \Big(\sum_{i>k}\sigma_i^2\Big)^{1/2}.$$

Storage for $A_k$: $k(m + n + 1)$ numbers instead of $mn$.

### Prediction (write this *before* running)

- Which of photo / checkerboard / random noise needs the most singular values for a good approximation?
- What does the rank-1 approximation of the photo look like?

*(fill in)*

In [ ]:
from skimage import data

photo = data.camera().astype(float) / 255            # 512 × 512 grayscale
# photo = plt.imread("my_photo.jpg").mean(axis=2)    # or use your own image
n = photo.shape[0]
blk = 64
checker = np.kron(np.indices((n // blk, n // blk)).sum(axis=0) % 2, np.ones((blk, blk))).astype(float)
noise = np.random.default_rng(0).random((n, n))
images = {"photo": photo, "checkerboard": checker, "noise": noise}

svds = {name: np.linalg.svd(img, full_matrices=False) for name, img in images.items()}

fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img, cmap="gray"); ax.set_title(name); ax.axis("off")
for name, (_, s, _) in svds.items():
    axes[3].semilogy(s / s[0], label=name)
axes[3].set_ylim(1e-18, 2); axes[3].set_xlabel("i"); axes[3].set_ylabel("σᵢ / σ₁")
axes[3].legend(); axes[3].grid(alpha=0.3); axes[3].set_title("singular values")
plt.show()

In [ ]:
ks = [1, 5, 20, 50, 100]

def show_low_rank(name):
    img = images[name]
    U, s, Vt = svds[name]
    fig, axes = plt.subplots(1, len(ks) + 1, figsize=(20, 3.9))
    axes[0].imshow(img, cmap="gray"); axes[0].set_title(f"{name}: original")
    for ax, k in zip(axes[1:], ks):
        Ak = (U[:, :k] * s[:k]) @ Vt[:k]
        rel = np.linalg.norm(img - Ak) / np.linalg.norm(img)
        ax.imshow(Ak, cmap="gray", vmin=img.min(), vmax=img.max())
        ax.set_title(f"k = {k}\nrel. error {rel:.3f}, storage {k * (sum(img.shape) + 1) / img.size:.1%}")
    for ax in axes:
        ax.axis("off")
    plt.show()

for name in images:
    show_low_rank(name)

### Verify Eckart–Young numerically

In [ ]:
for name, img in images.items():
    U, s, Vt = svds[name]
    ok2, okF = True, True
    for k in ks:
        Ak = (U[:, :k] * s[:k]) @ Vt[:k]
        e2, eF = np.linalg.norm(img - Ak, 2), np.linalg.norm(img - Ak, "fro")
        ok2 &= np.isclose(e2, s[k], rtol=1e-6, atol=1e-10 * s[0])
        okF &= np.isclose(eF, np.sqrt(np.sum(s[k:] ** 2)), rtol=1e-6, atol=1e-10 * s[0])
        print(f"{name:>12}  k={k:>3}:  ‖A−A_k‖₂ = {e2:.5f}   σ_k+1 = {s[k]:.5f}   |   "
              f"‖A−A_k‖_F = {eF:.5f}   √Σσ² = {np.sqrt(np.sum(s[k:]**2)):.5f}")
    check(f"{name}: ‖A − A_k‖₂ = σ_k+1 for all k", ok2)
    check(f"{name}: ‖A − A_k‖_F = (Σ_{{i>k}} σᵢ²)^½ for all k", okF)
    print()

### Compare compression across the three images

In [ ]:
kk = np.arange(0, 201)
fig, ax = plt.subplots(figsize=(9, 5))
for name, (_, s, _) in svds.items():
    tail = np.sqrt(np.maximum(np.sum(s**2) - np.cumsum(np.r_[0, s**2])[kk], 0))
    ax.semilogy(kk, tail / np.sqrt(np.sum(s**2)) + 1e-17, label=name)
ax.set_xlabel("k"); ax.set_ylabel("relative Frobenius error ‖A − A_k‖_F / ‖A‖_F")
ax.legend(); ax.grid(alpha=0.3, which="both")
plt.show()

---
### Explain (hand-in — in your own words)

**1. Check the Eckart–Young errors [4.10].**

*Your answer.*

**2. Which image compresses best, and why?**
*(Hint: what is the exact rank of the checkerboard? For noise, how are the singular values spread out?)*

*Your answer.*

---
## 4.4 Sum of rank-1 pieces

**Task.** Using the matrix in Example 4.6, plot each term $\sigma_k u_k v_k^\top$ as a heatmap, together with the running sum.

$$A = \begin{bmatrix}2&1&0&1\\1&3&4&2\\0&1&3&3\end{bmatrix}$$

### Prediction (write this *before* running)

- Which entries of $A$ will the first piece $\sigma_1 u_1 v_1^\top$ capture best?
- How many pieces are needed to get $A$ back exactly?

*(fill in)*

In [ ]:
A = np.array([[2, 1, 0, 1], [1, 3, 4, 2], [0, 1, 3, 3]], dtype=float)
U, s, Vt = np.linalg.svd(A)
r = len(s)
pieces = [s[k] * np.outer(U[:, k], Vt[k]) for k in range(r)]
running = np.cumsum(pieces, axis=0)
vmax = np.abs(A).max()

def heat(ax, M, title):
    ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    for (i, j), val in np.ndenumerate(M):
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(2, r + 1, figsize=(17, 6))
for k in range(r):
    heat(axes[0, k], pieces[k], f"σ{k+1} u{k+1} v{k+1}ᵀ   (σ{k+1} = {s[k]:.3f})")
    heat(axes[1, k], running[k], f"running sum A{k+1}")
heat(axes[0, r], A, "A")
heat(axes[1, r], A - running[-1], "A − A₃")
plt.tight_layout(); plt.show()

### Verify numerically

In [ ]:
A1_notes = np.array([[0.217, 0.588, 0.923, 0.665], [0.902, 2.439, 3.832, 2.761], [0.69, 1.864, 2.929, 2.11]])
A2_notes = np.array([[1.888, 1.319, 0.198, 0.479], [1.157, 2.551, 3.721, 2.732], [-0.171, 1.487, 3.302, 2.206]])

check("σ = (6.909, 2.238, 1.501)  (notes)", np.allclose(s, [6.909, 2.238, 1.501], atol=1e-3))
check("every piece has rank 1", all(np.linalg.matrix_rank(P) == 1 for P in pieces))
check("A₁ matches the notes", np.allclose(running[0], A1_notes, atol=2e-3))
check("A₂ matches the notes", np.allclose(running[1], A2_notes, atol=2e-3))
check("A₃ = A", np.allclose(running[2], A))
check("‖A − A_k‖₂ = σ_k+1 for k = 1, 2", all(np.isclose(np.linalg.norm(A - running[k - 1], 2), s[k]) for k in [1, 2]))

### Schatten norms (Example 4.4 uses the same matrix)

In [ ]:
lam, W = np.linalg.eigh(A.T @ A)
sqrt_AtA = W @ np.diag(np.sqrt(np.clip(lam, 0, None))) @ W.T     # the PSD square root of AᵀA

spec, fro, nuc = np.linalg.norm(A, 2), np.linalg.norm(A, "fro"), np.linalg.norm(A, "nuc")
print(f"‖A‖₂ = {spec:.3f}   ‖A‖_F = {fro:.3f}   ‖A‖_N = {nuc:.3f}")
check("‖A‖₂ = σ₁ = 6.909", np.isclose(spec, s[0]) and np.isclose(spec, 6.909, atol=1e-3))
check("‖A‖_F = √Σσ² = √tr(AᵀA) = √Σaᵢⱼ² = 7.416",
      np.allclose([fro, np.sqrt(np.sum(s**2)), np.sqrt(np.trace(A.T @ A)), np.sqrt(np.sum(A**2))], 7.416, atol=1e-3))
check("‖A‖_N = Σσ = tr(√(AᵀA)) = 10.648",
      np.allclose([nuc, s.sum(), np.trace(sqrt_AtA)], 10.648, atol=1e-3))

---
### Explain (hand-in — in your own words)

**1. What does each rank-1 piece add?**
*(Hint: compare the heatmaps of the pieces with the error $A - A_k$. Why does the first piece capture the big block of large entries?)*

*Your answer.*

**2. Compute the three Schatten norms of Example 4.4 and check them against the singular values: $\|A\|_2 = \sigma_1$, $\|A\|_F = \sqrt{\sum_k\sigma_k^2} = \sqrt{\mathrm{tr}(A^\top A)}$, $\|A\|_N = \sum_k\sigma_k$.**

*Your answer.*